In [ ]:
# The following lines enable automatic reloading of modules in an IPython/Jupyter environment.
# They work exactly like the commented lines below, but avoid errors when not running in such an environment.
# %load_ext autoreload
# %autoreload 2

try:
    # Only defined inside IPython/Jupyter
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except (NameError, AttributeError):
    # Not running in IPython → just ignore
    pass


In [ ]:
# Initializing answer variable
answer = {}


In [ ]:
# Some libs that we will use
import torch
import random
import numpy as np
import json_tricks
import lovely_tensors as lt
from pprint import pprint

# Making tensor printouts better
lt.monkey_patch()

# Adding sources to the pythonpath
import sys
root_path = '../../../..'
sys.path.append(root_path)

import dotenv
dotenv.load_dotenv(dotenv.find_dotenv(root_path + '/.env'))

# Importing sources of our project
import src
import src.utils as utils


In [ ]:
src.utils.seed.seed_all(0)
# ^^^ This command should be edited

# Seed checks
seed_check = {
    'python.random': [random.random() for _ in range(4)],
    'python.randint': [random.randint(0, 9) for _ in range(4)],
    'numpy.random.rand': np.random.rand(2, 3),
    'numpy.random.randn': np.random.randn(2, 3),
    'numpy.random.randint': np.random.randint(0, 10, size=(2, 3)),
    'torch.rand': torch.rand(2, 3),
    'torch.randn': torch.randn(2, 3),
    'torch.randint': torch.randint(0, 10, (2, 3)),
    'torch.backends.cudnn.deterministic': torch.backends.cudnn.deterministic,
    'torch.cuda.is_available': torch.cuda.is_available(),
}

if seed_check['torch.cuda.is_available']:
    seed_check['torch.cuda.initial_seed'] = torch.cuda.initial_seed()
    seed_check['torch.cuda.rand'] = torch.rand(2, 3, device='cuda').cpu()
    seed_check['torch.cuda.randn'] = torch.randn(2, 3, device='cuda').cpu()
    seed_check['torch.cuda.randint'] = torch.randint(0, 10, (2, 3), device='cuda').cpu()

answer['seed_check'] = src.utils.detach_copy(seed_check)

print("!!! THE VALUES BELOW SHOULD BE STABLE FOR DIFFERENT RUNS !!!")
print()
pprint(seed_check)

In [ ]:
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False)
# ^^^ This class should be implemented in this task

In [ ]:
train_sample = MNIST_train[0]
valid_sample = MNIST_valid[0]

X_train = train_sample['image']
X_valid = valid_sample['image']
y_train = train_sample['label']
y_valid = valid_sample['label']


In [ ]:
## This checks are for dataset verification
answer['X_train.dtype'] = str(X_train.dtype)
answer['y_train.dtype'] = str(y_train.dtype)
answer['X_valid.dtype'] = str(X_valid.dtype)
answer['y_valid.dtype'] = str(y_valid.dtype)
answer['X_train.shape'] = X_train.shape
answer['X_valid.shape'] = X_valid.shape
answer['y_train.shape'] = y_train.shape
answer['y_valid.shape'] = y_valid.shape
answer['X_train.mean'] = float(X_train.mean())
answer['y_train.mean'] = float(y_train.float().mean())
answer['X_valid.mean'] = float(X_valid.mean())
answer['y_valid.mean'] = float(y_valid.float().mean())

print('\nX_train dtype:', X_train.dtype, '\nX_valid dtype:', X_valid.dtype, '\ny_train dtype', y_train.dtype, '\ny_valid dtype:', y_valid.dtype)
print('\nX_train shape:', X_train.shape, '\nX_valid shape', X_valid.shape, '\ny_train shape:', y_train.shape, '\ny_valid shape:', y_valid.shape)

print()
print('X train:', X_train)
print('X_valid:', X_valid)
print('y_train:', y_train)
print('y_valid:', y_valid)


Neural networks are usually trained with floating-point tensors, most commonly `float32` or `float16`. In this exercise, the dataset class should return images as `float32` tensors.


Let us visualize one training sample.


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(X_train)
plt.show()
print(y_train)


## Batch Updating Protocol (stays the same)

## Motivation

Before we build the model, let us agree on how information moves through the training pipeline. A dataset item starts as a small dictionary, for example `{"image": ..., "label": ...}`. When the dataloader combines many items into a mini-batch, the training loop wraps that batch into a nested dictionary:

```python
batch = {'data': original_batch}
```

## Batch Sections

The `data` section contains values that came from the dataset, such as `batch['data']['image']` and `batch['data']['label']`. The model should not overwrite these values. Instead, it adds new sections as it works:

- `batch['signals']`: raw differentiable tensors produced by the network, such as logits in `batch['signals']['output']`;
- `batch['postprocessed']`: non-loss-facing predictions or summaries, such as predicted class indices in `batch['postprocessed']['class']`;
- `batch['loss']`: the scalar loss tensor computed by the loss function during training.

## Responsibility Split

This update-in-place style lets the model, loss function, metric function, and training loop communicate through one shared object while still keeping their responsibilities separate. The model writes signals and predictions, the loss reads the batch and returns a scalar, metrics read the batch and return aggregated counts, and the training loop controls optimization.


# Task 1: Build a Fully Convolutional ResNet

## Goal

Edit [src/models/convolutional/classifier.py](../../../../src/models/feedforward/simple_fcnn.py).

The idea is the same as in case of Fully Connected networks.
The algorithm is:
- you create a sequence of blocks with `encoder` (or `stem`) and `decoder` (in our case it can be referred to as `classifier`)
- after that you experiment with different blocks, including the following options:
    - `Residual` connections
    - batch normalizations (in case of convolutional networks, use `torch.nn.BatchNorm2d`)
    - different activations
    - instead of `Linear` blocks, there will be `torch.nn.Conv2d` blocks
    - once in a while there will be either `torch.nn.MaxPool2d` blocks or strides in `torch.nn.Conv2d` blocks

The modules now are separated in levels, after each a block with downsampling should be located (so, in each level there are `n_blocks[lvl] + 1` blocks in fact).

Note that the classifier should be predecessored by global average pooling (averaging of the signal via all the spatial dimensions).

Note that in the case of convolutional networks, we will use Poolings or Strides, and in these blocks, where this happens, we will not be able to perform the proper residual connections, thus we will need to do something clever in this case.

What can go wrong in case of these blocks?
- firstly, spacial dimensions are different after pools/strides
- secondly, usually when pooling/strides happens, we increase number of channels
- in case nothing like that happens, we do $\mathbf x + NetInNet(\mathbf x)$
- in case there is mismatch, we do bypass: 
    
    $Adapter(\mathbf x) + NetInNet^*(\mathbf x)$
    
    where $Adapter$ is usually just a single convolution (`Conv2d 1x1/2 n_in -> n_out`). Note stride 2.

    The main idea that it should be damn simple: no activation, no fan in, and we are sure that we do it only very limited number of times in our network -- then gradient vanishing does not matter than much.

!!! NOTE !!! 
Interestingly, for the convolutional networks, usage of batch normalization is way more important than in fully connected networks

!!! NOTE !!!
We are very close to the production-level networks.
Here are the difference between our setup and production-level setups.

- MNIST is 1 channel, while images have 3 channels
- MNIST is 28x28, while HD images usually are 1024x1024 and more

The first issue is very easy to resolve, while the second often requires preliminary compression of the image.
There are several ways to compress images:
- Stem layer (usually Conv2d with very large kernel (7x7) and a stride 2-4)
- Fourier Transform (never saw anybody doing that though, but looks very intersting)
- pretrained VAE encoder, very often used in Pix2Pix diffusion models

## Parts To Implement

In this task the following modules should be implemented:
- `FullyConvolutionalNN`
- `Bottleneck`

Implement three parts:
- `__init__`, which creates the modules of the neural network. Remember to initialize the parent `torch.nn.Module` with `super().__init__()`.
- `__forward_kernel`, which receives an image tensor, flattens it from `[batch, 28, 28]` to `[batch, 784]`, passes it through the fully connected layers, and returns raw logits. This method is also used for the model visualization.
- `forward`, which receives the nested batch dictionary, reads `batch['data']['image']`, calls `__forward_kernel`, stores logits in `batch['signals']['output']`, calls `argmax` to get predicted classes, and stores them in `batch['postprocessed']['class']`.


Let us check that your network runs and produces logits with the expected shape.


In [ ]:
fcnn = src.models.convolutional.sequential.FullyConvolutionalNN(
    in_channels=1,
    mid_channels=[16, 32, 64, 128],
    out_channels=10,
    block = lambda in_channels, out_channels: src.models.convolutional.sequential.ResidualBottleneck(in_channels, out_channels),
    n_blocks=[2, 2, 2, 2])
# ^^^ This class should be implemented in this task

# Here we intialize the network so that the initial state is predictable
utils.deterministic_init(fcnn)

# Here we are creating a sample batch with the same keys as MNISTSimpleDataset
check_batch = {'data': {'image': torch.randn(10, 28, 28), 'label': torch.randint(low=0, high=10, size=(10,))}}

# And checking how it traverses through the network
check_output = fcnn(check_batch)

print('output:', check_batch['signals']['output'].shape)
print('classes:', check_batch['postprocessed']['class'].shape)

# And saving the result
pprint(check_batch)
answer['check_batch'] = check_batch


## Model Graph

Let us also draw the model graph. A visual graph is often helpful when you want to check whether the signal flows through the modules in the order you intended.

The utility [src/utils/model_visualisation.py](../../../../src/utils/model_visualisation.py) uses a small internal adapter so that `torchview` draws only the tensor path through `__forward_kernel`. That keeps the graph focused on the neural-network layers rather than on the nested batch dictionary and postprocessing keys.

## Empty Backbone Note

One important detail: `backbone` and `classifier` are module names, while the graph is built from the actual tensor operations. If the current model uses `channels=[784]`, then `backbone` is an empty `Sequential`, so there is no hidden-layer operation for the graph to show. In that case we draw a small non-empty example network below just for visualization.


In [ ]:
print("Network structure:")
print(fcnn)

# Task 2: Create the Loss Function, Optimizer, Scheduler, and Metric (everything stays the same as in classification task, you can just copy-paste)


In [ ]:
import torchmetrics

def loss(batch):
    return batch['signals']['output'].abs().sum()

losses = {'ce': loss}

def accuracy(batch):
    return 0, 1

metrics = {'accuracy': accuracy}
optimizer = ...
scheduler = ...

# ^^^ These variables should be set up in this task
## YOUR CODE HERE


In [ ]:
answer['loss_names'] = list(losses.keys())
answer['metric_names'] = list(metrics.keys())

answer['optimizer_class'] = optimizer.__class__.__name__
answer['optimizer_defaults'] = optimizer.defaults
answer['optimizer_param_groups_count'] = len(optimizer.param_groups)
answer['optimizer_lrs'] = [group['lr'] for group in optimizer.param_groups]

answer['scheduler_class'] = scheduler.__class__.__name__
answer['scheduler_optimizer_class'] = scheduler.optimizer.__class__.__name__

answer['test_loss_val'] = {f_name: function(check_batch) for f_name, function in losses.items()}
answer['test_metrics_val'] = {f_name: function(check_batch) for f_name, function in metrics.items()}

print(losses)
print(optimizer)
print(scheduler)
print(metrics)


# Create DataLoaders for MNIST (everything stays the same as in classification task, you can just copy-paste)

In [ ]:
from tqdm import trange, tqdm
import os

batch_size = ...
train_dl = ...
test_dl = ...

# ^^^ These variables should be set up in this task
## YOUR CODE HERE

# Sanity checking that the dataloaders work
train_batch = next(iter(train_dl))
valid_batch = next(iter(valid_dl))

pprint(train_batch)
pprint(valid_batch)

# Storing the answers
answer['train_batch'] = str(train_batch)
answer['valid_batch'] = str(valid_batch)

# Task 3: Create the networks

Create the following set of networks:
- convolutional network 
- convolutional network with prenorm
- convolutional network with resnet
- convolutional network with resnet and batchnorm

Check their gradients in the initial states

In [ ]:
def fcnn(blocks=10):
    network = src.models.convolutional.sequential.FullyConvolutionalNN(
        in_channels=1,
        mid_channels=[16, 32, 64, 128],
        out_channels=10,
        block = lambda in_channels, out_channels: src.models.convolutional.sequential.ResidualBottleneck(
            in_channels, 
            out_channels,
            residual=False),
        n_blocks=[blocks, blocks, blocks, blocks])
    
    ## YOUR CODE HERE

    return network


def fcnn_bn(blocks=10):
    network = src.models.convolutional.sequential.FullyConvolutionalNN(
        in_channels=1,
        mid_channels=[16, 32, 64, 128],
        out_channels=10,
        block = lambda in_channels, out_channels: src.models.convolutional.sequential.ResidualBottleneck(
            in_channels, out_channels,
            residual=False),
        n_blocks=[blocks, blocks, blocks, blocks])
    
    ## YOUR CODE HERE


    return network


def fcnn_resnet(blocks=10):
    network = src.models.convolutional.sequential.FullyConvolutionalNN(
        in_channels=1,
        mid_channels=[16, 32, 64, 128],
        out_channels=10,
        block = lambda in_channels, out_channels: src.models.convolutional.sequential.ResidualBottleneck(
            in_channels, 
            out_channels,
            residual=False),
        n_blocks=[blocks, blocks, blocks, blocks])
    
    ## YOUR CODE HERE


    return network


def fcnn_resnet_bn(blocks=10):
    network = src.models.convolutional.sequential.FullyConvolutionalNN(
        in_channels=1,
        mid_channels=[16, 32, 64, 128],
        out_channels=10,
        block = lambda in_channels, out_channels: src.models.convolutional.sequential.ResidualBottleneck(
            in_channels, 
            out_channels,
            residual=False),
        n_blocks=[blocks, blocks, blocks, blocks])
    
    ## YOUR CODE HERE


    return network

In [ ]:
gradient_demo_batch = next(iter(valid_dl))


def summarize_parameter_stats(model, loss_value):
    modules = []

    for name, module in model.named_modules():
        if name == '':
            continue

        weights = []
        grads = []
        for parameter in module.parameters(recurse=False):
            weights.append(parameter.detach().abs().flatten())
            if parameter.grad is not None:
                grads.append(parameter.grad.detach().abs().flatten())

        if not weights:
            continue

        weight_values = torch.cat(weights)
        grad_values = torch.cat(grads) if grads else torch.zeros(1)

        modules.append({
            'name': name,
            'max_weight': float(weight_values.max()),
            'mean_weight': float(weight_values.mean()),
            'max_grad': float(grad_values.max()),
            'mean_grad': float(grad_values.mean()),
        })

    return {
        'loss': float(loss_value.detach()),
        'modules': modules,
    }


def plot_fresh_signal_stats(title, model_factory, seed=0):
    src.utils.seed.seed_all(seed)
    model = model_factory()
    model.zero_grad(set_to_none=True)

    batch = {'data': gradient_demo_batch}
    model.train()
    model(batch)
    signal_stats_loss = losses['ce'](batch)
    signal_stats_loss.backward()
    stats = summarize_parameter_stats(model, signal_stats_loss)

    print(title)
    src.utils.model_visualization.plot_signal_stats(model, {'data': gradient_demo_batch})
    return stats


answer['signal_stats'] = {}

answer['signal_stats']['convolutional'] = plot_fresh_signal_stats(
    'Fresh network: convolutional',
    fcnn)

answer['signal_stats']['convolutional + batchnorm'] = plot_fresh_signal_stats(
    'Fresh network: convolutional + batchnorm',
    fcnn_bn)

answer['signal_stats']['resnet'] = plot_fresh_signal_stats(
    'Fresh network: resnet',
    fcnn_resnet)

answer['signal_stats']['resnet + batchnorm'] = plot_fresh_signal_stats(
    'Fresh network: resnet + batchnorm',
    fcnn_resnet_bn)

In [ ]:
json_tricks.dump(utils.torch_to_numpy(answer), '.answer.json')

# Task 6: Experiment Time


!!! DO NOT CHECK THIS CODE !!!

Now train the network for 20 epochs. You should see way better accuracy than in case of Fully Connected network


In [ ]:
# for network_constructor in [fcnn, fcnn_bn, fcnn_resnet, fcnn_resnet_bn]:
#     network = network_constructor(2)

#     optimizer = torch.optim.AdamW(network.parameters(), lr=1.0e-3)
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

#     n_epochs = 20

#     training_logger = src.utils.mlflow.prepare_mlflow_logger(
#         str(network_constructor.__name__))

#     shallow_training_history = src.train_loop.train_model(
#         network,
#         n_epochs,
#         train_dl,
#         valid_dl,
#         losses,
#         optimizer,
#         metrics=metrics,
#         scheduler=scheduler,
#         mlflow_logger=training_logger,
#         run_name='shallow_network')

#     train_loss_history = shallow_training_history['train_loss']['ce']
#     valid_loss_history = shallow_training_history['valid_loss']['ce']
#     train_acc_history = shallow_training_history['train_metrics']['accuracy']
#     valid_acc_history = shallow_training_history['valid_metrics']['accuracy']

#     training_logger.end()
